# Лабораторная работа №1 — KNN

Сравниваем четыре варианта KNN на классификации писем и прогнозе температуры. KNN не строит отдельную модель при обучении: он запоминает обучающую выборку и использует ближайшие примеры при прогнозе.

## 1. Выбор начальных условий

**Классификация — [Spambase](https://archive.ics.uci.edu/dataset/94/spambase).** Почтовый фильтр по 57 числовым признакам письма (частоты слов и символов, статистики заглавных букв) решает, является ли оно спамом (`1`) или нет (`0`). Набор подходит для задачи фильтрации: признаки можно извлечь из письма при его получении. После исключения дублей и конфликтующих меток используем стратифицированное разбиение 80/20 (`random_state=42`). Основная метрика — **F1-macro**, которая одинаково учитывает оба класса, хотя спам встречается реже. Дополнительно считаем **accuracy**, то есть долю верных ответов.

**Регрессия — [SML2010](https://archive.ics.uci.edu/dataset/274/sml2010).** Для контроля микроклимата прогнозируем температуру в столовой через 15 минут по 18 текущим признакам (температура, влажность, освещённость и другие показания датчиков). Метка берётся строго по времени `t + 15 мин`: используем только пары с таким интервалом. Обучение — первый период, тест — второй, без перемешивания. Основная метрика — **MAE**, средняя абсолютная ошибка в °C. Дополнительно считаем **RMSE**, которая сильнее реагирует на большие ошибки.

Лицензии, источники и правила подготовки описаны в [протоколе оценки](../docs/data-protocol.md). Уникальность наборов в группе нужно подтвердить отдельно.

Загружаем общие выборки и выводим их размеры. Эти же разбиения используются в остальных лабораторных.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from data_setup import DINING_TEMPERATURE, load_sml2010, load_spambase, sml2010_cv, spambase_cv

Xc_train, Xc_test, yc_train, yc_test = load_spambase()
Xr_train, Xr_test, yr_train, yr_test = load_sml2010()
print(f"Spambase: {Xc_train.shape} / {Xc_test.shape}")
print(f"SML2010:  {Xr_train.shape} / {Xr_test.shape}")

Spambase: (3363, 57) / (841, 57)
SML2010:  (2763, 18) / (1370, 18)


## 2. Создание бейзлайна и оценка качества

Обучаем классификатор и регрессор sklearn с `k=5`, евклидовым расстоянием, равными весами соседей и без масштабирования. Функции оценки будут одинаковыми для всех вариантов. Для прогноза температуры также показываем простой ориентир — текущую температуру (`ŷ(t + 15) = y(t)`); в CSV с вариантами KNN он не входит.

In [2]:
def classification_metrics(y, prediction):
    return {"f1_macro": f1_score(y, prediction, average="macro"), "accuracy": accuracy_score(y, prediction)}


def regression_metrics(y, prediction):
    return {"mae": mean_absolute_error(y, prediction), "rmse": root_mean_squared_error(y, prediction)}


clf_base = KNeighborsClassifier().fit(Xc_train, yc_train)
reg_base = KNeighborsRegressor().fit(Xr_train, yr_train)
classification_scores = {"sklearn_base": classification_metrics(yc_test, clf_base.predict(Xc_test))}
regression_scores = {"sklearn_base": regression_metrics(yr_test, reg_base.predict(Xr_test))}
persistence = regression_metrics(yr_test, Xr_test[DINING_TEMPERATURE])

display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict({**regression_scores, "текущая температура": persistence}, orient="index"))

,f1_macro,accuracy
sklearn_base,0.774329,0.785969


,mae,rmse
sklearn_base,3.102876,3.727811
текущая температура,0.116360,0.133181


## 3. Улучшение бейзлайна

### 3a. Гипотезы

1. Стандартизация может помочь: при евклидовом расстоянии признаки большого масштаба иначе сильнее влияют на выбор соседей.
2. `k=5` может быть не лучшим: меньшее `k` лучше отражает локальные различия, большее сглаживает отдельные выбросы. Проверяем `k ∈ {3, 5, 11, 21}`.

### 3b. Проверка гипотез

Для каждой задачи перебираем комбинации `k` и масштабирования. Подбор идёт **только на обучающей части**. Для классификации используем три стратифицированных фолда и F1-macro, для регрессии — три хронологических фолда с зазором и MAE. `Pipeline` обучает `StandardScaler` отдельно внутри каждого фолда. В таблицах показываем средние CV-метрики; отрицательный MAE, который sklearn максимизирует при поиске, отображаем как обычную ошибку.

In [3]:
def search_knn(model, cv, scoring, X, y):
    pipeline = Pipeline([("scale", "passthrough"), ("knn", model)])
    grid = {"scale": ["passthrough", StandardScaler()], "knn__n_neighbors": [3, 5, 11, 21]}
    return GridSearchCV(pipeline, grid, cv=cv, scoring=scoring).fit(X, y)


def cv_table(search, metric, sign=1):
    results = search.cv_results_
    return pd.DataFrame({
        "масштабирование": ["нет" if scale == "passthrough" else "StandardScaler" for scale in results["param_scale"]],
        "k": results["param_knn__n_neighbors"],
        metric: sign * results["mean_test_score"],
    }).sort_values(metric, ascending=(sign == -1))


clf_search = search_knn(KNeighborsClassifier(), spambase_cv(), "f1_macro", Xc_train, yc_train)
reg_search = search_knn(KNeighborsRegressor(), sml2010_cv(), "neg_mean_absolute_error", Xr_train, yr_train)
display(cv_table(clf_search, "CV F1-macro"))
display(cv_table(reg_search, "CV MAE, °C", sign=-1))

,масштабирование,k,CV F1-macro
3,StandardScaler,5,0.886613
1,StandardScaler,3,0.884971
5,StandardScaler,11,0.876555
7,StandardScaler,21,0.872289
0,нет,3,0.766262
2,нет,5,0.760998
4,нет,11,0.752504
6,нет,21,0.728136


,масштабирование,k,"CV MAE, °C"
5,StandardScaler,11,0.839432
3,StandardScaler,5,0.844771
7,StandardScaler,21,0.854750
1,StandardScaler,3,0.855524
6,нет,21,1.881825
4,нет,11,1.917604
2,нет,5,1.927605
0,нет,3,1.955799


### 3c. Улучшенный бейзлайн

Берём конфигурации с лучшими средними CV-метриками. `GridSearchCV` уже переобучил их на всей обучающей части. Показываем выбранные параметры, оцениваем обе модели на тесте и сравниваем их с пунктом 2.

In [4]:
print("Spambase:", clf_search.best_params_)
print("SML2010:", reg_search.best_params_)
classification_scores["sklearn_improved"] = classification_metrics(yc_test, clf_search.predict(Xc_test))
regression_scores["sklearn_improved"] = regression_metrics(yr_test, reg_search.predict(Xr_test))
display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict(regression_scores, orient="index"))

Spambase: {'knn__n_neighbors': 5, 'scale': StandardScaler()}
SML2010: {'knn__n_neighbors': 11, 'scale': StandardScaler()}


,f1_macro,accuracy
sklearn_base,0.774329,0.785969
sklearn_improved,0.912908,0.916766


,mae,rmse
sklearn_base,3.102876,3.727811
sklearn_improved,1.298335,1.649461


**Выводы по улучшению.** На Spambase стандартизация подняла тестовый F1-macro с 0,7743 до 0,9129; по кросс-валидации `k=5` остался лучшим среди проверенных значений. На SML2010 стандартизация снизила CV MAE при `k=5` с 1,928 до 0,845 °C, а выбор `k=11` снизил её ещё немного, до 0,839 °C. На тесте улучшенный KNN снизил MAE с 3,103 до 1,298 °C, но прогноз по текущей температуре оказался точнее (MAE 0,116 °C). Поэтому на данном горизонте выигрыш относительно базового KNN не делает улучшенный KNN лучшим практическим выбором.

## 4. Имплементация алгоритма машинного обучения

### 4.1. Базовая реализация

Собственный KNN хранит обучающие данные в `fit`. Для каждой новой точки `predict` считает квадраты евклидовых расстояний до всех обучающих примеров и берёт `k` ближайших. Квадратный корень не нужен: он не меняет порядок расстояний. Для классификации голосуют соседи, при равенстве голосов выбирается меньшая метка. Для регрессии усредняем целевые значения соседей. При равных расстояниях сохраняем порядок обучающих строк.

In [5]:
class KNN:
    def __init__(self, k=5, classification=True):
        self.k = k
        self.classification = classification

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        predictions = []
        for row in np.asarray(X, dtype=float):
            distances = ((self.X_train - row) ** 2).sum(axis=1)
            neighbors = np.argsort(distances, kind="stable")[:self.k]
            targets = self.y_train[neighbors]
            if self.classification:
                labels, counts = np.unique(targets, return_counts=True)
                predictions.append(labels[counts.argmax()])
            else:
                predictions.append(targets.mean())
        return np.asarray(predictions)

Проверяем голосование, разрешение ничьей и среднее на маленьких примерах. Затем используем ту же конфигурацию, что в пункте 2, и оцениваем обе задачи на тех же тестовых выборках.

In [6]:
assert KNN(k=3).fit([[0], [1], [4]], [1, 1, 0]).predict([[1]])[0] == 1
assert KNN(k=2).fit([[0], [1], [4]], [1, 0, 1]).predict([[1]])[0] == 0
assert KNN(k=2, classification=False).fit([[0], [1], [4]], [10, 20, 50]).predict([[1]])[0] == 15

clf_own = KNN().fit(Xc_train, yc_train)
reg_own = KNN(classification=False).fit(Xr_train, yr_train)
classification_scores["custom_base"] = classification_metrics(yc_test, clf_own.predict(Xc_test))
regression_scores["custom_base"] = regression_metrics(yr_test, reg_own.predict(Xr_test))
base_variants = ["sklearn_base", "custom_base"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[base_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[base_variants])

,f1_macro,accuracy
sklearn_base,0.774329,0.785969
custom_base,0.774329,0.785969


,mae,rmse
sklearn_base,3.102876,3.727811
custom_base,3.102876,3.727811


**Сравнение с пунктом 2.** На этих тестовых выборках базовый NumPy KNN совпал со sklearn по обеим метрикам классификации и обеим метрикам регрессии с точностью вывода. При равных расстояниях на границе `k` выбор среди соседей может отличаться в других данных.

### 4.2. Улучшенная реализация

Переносим `k` и решение о стандартизации из пункта 3c без повторного подбора. Если выбрана стандартизация, обучаем новый `StandardScaler` только на обучающих признаках. Сам поиск соседей, голосование и усреднение остаются собственными. Сравниваем с улучшенными sklearn-моделями.

In [7]:
def predict_improved(search, X_train, X_test, y_train, *, classification):
    params = search.best_params_
    if params["scale"] != "passthrough":
        scaler = StandardScaler().fit(X_train)
        X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
    model = KNN(k=params["knn__n_neighbors"], classification=classification).fit(X_train, y_train)
    return model.predict(X_test)


classification_scores["custom_improved"] = classification_metrics(
    yc_test, predict_improved(clf_search, Xc_train, Xc_test, yc_train, classification=True),
)
regression_scores["custom_improved"] = regression_metrics(
    yr_test, predict_improved(reg_search, Xr_train, Xr_test, yr_train, classification=False),
)
improved_variants = ["sklearn_improved", "custom_improved"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[improved_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[improved_variants])

,f1_macro,accuracy
sklearn_improved,0.912908,0.916766
custom_improved,0.912908,0.916766


,mae,rmse
sklearn_improved,1.298335,1.649461
custom_improved,1.298335,1.649461


**Сравнение с пунктом 3.** После переноса `StandardScaler` и выбранных `k` улучшенные варианты совпали со sklearn по всем четырём метрикам с точностью вывода. Для своего KNN стандартизация дала тот же выигрыш относительно базовой версии, но в регрессии простой прогноз по текущей температуре всё равно точнее.

Сохраняем восемь строк для итогового ноутбука: четыре варианта на каждую задачу. В каждой строке заполнены только метрики её задачи.

In [8]:
variants = ["sklearn_base", "sklearn_improved", "custom_base", "custom_improved"]
rows = []
for task, scores in [("classification", classification_scores), ("regression", regression_scores)]:
    for variant in variants:
        rows.append({"lab": 1, "task": task, "algorithm": "KNN", "variant": variant, **scores[variant]})

results = pd.DataFrame(rows)
results.to_csv(root / "results" / "lab01.csv", index=False, float_format="%.8f")
for task in results["task"].unique():
    display(results.loc[results["task"] == task].dropna(axis=1, how="all").reset_index(drop=True))

,lab,task,algorithm,variant,f1_macro,accuracy
0,1,classification,KNN,sklearn_base,0.774329,0.785969
1,1,classification,KNN,sklearn_improved,0.912908,0.916766
2,1,classification,KNN,custom_base,0.774329,0.785969
3,1,classification,KNN,custom_improved,0.912908,0.916766


,lab,task,algorithm,variant,mae,rmse
0,1,regression,KNN,sklearn_base,3.102876,3.727811
1,1,regression,KNN,sklearn_improved,1.298335,1.649461
2,1,regression,KNN,custom_base,3.102876,3.727811
3,1,regression,KNN,custom_improved,1.298335,1.649461
